# Hugging Face Transformers: From Basics to Advanced

**Purpose:** Learn the `transformers` library so you can read and use pretrained AI models (BERT, GPT, T5, LLaMA, etc.) in ML/AI projects.

The Hugging Face `transformers` library is how the modern AI world accesses pretrained models. If you see a project using BERT for NLP, Whisper for speech, or ViT for images — it's almost certainly using this library.

**How this notebook is organized:**
1. The Big Picture — What This Library Does
2. Pipelines — The Easiest Way (1 line of code)
3. Tokenizers — How Text Becomes Numbers
4. Models — Loading and Using Pretrained Models
5. Model + Tokenizer Together — The Real Pattern
6. Fine-Tuning with `Trainer`
7. Fine-Tuning with Native PyTorch
8. Model Hub — Finding and Sharing Models
9. Generation — Text Generation with LLMs
10. Advanced: Custom Heads, LoRA, and Quantization Patterns
11. Real Patterns You'll See in AI Code

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import transformers
import torch
import numpy as np
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version:      {torch.__version__}")

## 1. The Big Picture

The `transformers` library has three core concepts:

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  Tokenizer   │ ──▶ │    Model     │ ──▶ │  Post-proc   │
│              │     │ (pretrained) │     │  (decode,     │
│ text → IDs   │     │ IDs → logits │     │   softmax)   │
└──────────────┘     └──────────────┘     └──────────────┘
```

- **Tokenizer**: Converts text to numbers (token IDs) and back
- **Model**: A pretrained neural network (BERT, GPT, T5, etc.)
- **Pipeline**: Wraps tokenizer + model + post-processing into one easy call

**The `Auto` classes** (`AutoTokenizer`, `AutoModel`, etc.) automatically detect which architecture to use based on the model name. This is why you'll see `Auto*` everywhere — it's the universal way to load any model.

## 2. Pipelines — The Easiest Way

`pipeline()` is the highest-level API. One line gives you a working AI model. You'll see this in demos, prototypes, and quick experiments.

In [ ]:
# Sentiment analysis — classify text as positive/negative
classifier = pipeline("sentiment-analysis")

result = classifier("I love this library! It makes AI so easy.")
print("Sentiment:", result)

# Batch input
results = classifier([
    "This movie was amazing, best I've ever seen!",
    "Terrible experience. I want a refund.",
    "It was okay, nothing special."
])
for text, res in zip(["amazing", "terrible", "okay"], results):
    print(f"  {text:>10}: {res['label']} ({res['score']:.2%})")

In [ ]:
# Named Entity Recognition (NER) — identify people, places, organizations
ner = pipeline("ner", aggregation_strategy="simple")

entities = ner("Elon Musk founded SpaceX in Hawthorne, California in 2002.")
print("Named Entities:")
for e in entities:
    print(f"  {e['word']:20s} → {e['entity_group']:10s} (score: {e['score']:.2%})")

In [ ]:
# Question Answering — extract answers from a context paragraph
qa = pipeline("question-answering")

context = """
The Hugging Face Transformers library was created in 2018 by Thomas Wolf 
and the Hugging Face team. It provides thousands of pretrained models for 
Natural Language Processing, Computer Vision, and Audio tasks. The library 
supports both PyTorch and TensorFlow backends.
"""

answer = qa(question="Who created the Transformers library?", context=context)
print(f"Q: Who created the Transformers library?")
print(f"A: {answer['answer']} (score: {answer['score']:.2%})")

answer2 = qa(question="What tasks does it support?", context=context)
print(f"\nQ: What tasks does it support?")
print(f"A: {answer2['answer']} (score: {answer2['score']:.2%})")

In [ ]:
# More pipelines — text generation, summarization, fill-mask, zero-shot

# Fill-mask — predict a missing word (BERT-style)
fill_mask = pipeline("fill-mask")
results = fill_mask("The capital of France is [MASK].")
print("Fill-mask (top 3):")
for r in results[:3]:
    print(f"  {r['token_str']:>10s}: {r['score']:.2%}")

In [ ]:
# Zero-shot classification — classify without training!
# Uses NLI (natural language inference) model under the hood
zero_shot = pipeline("zero-shot-classification")

result = zero_shot(
    "The stock market crashed today after the Fed raised interest rates.",
    candidate_labels=["finance", "sports", "technology", "politics"]
)
print("Zero-shot classification:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:>12s}: {score:.2%}")

# Available pipeline tasks:
# "sentiment-analysis"         — positive/negative
# "ner"                        — named entity recognition
# "question-answering"         — extractive QA
# "fill-mask"                  — predict masked tokens (BERT)
# "text-generation"            — generate text (GPT)
# "text2text-generation"       — seq2seq (T5, BART)
# "summarization"              — summarize long text
# "translation_en_to_fr"       — translation
# "zero-shot-classification"   — classify without training
# "feature-extraction"         — get embeddings
# "text-classification"        — general classification

## 3. Tokenizers — How Text Becomes Numbers

Tokenizers are the bridge between human text and model input. Understanding tokenization is essential for debugging and reading transformer code.

In [ ]:
# Load a tokenizer — AutoTokenizer picks the right one for the model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Basic tokenization: text → tokens → IDs
text = "Hello, how are you doing today?"

tokens = tokenizer.tokenize(text)        # Text → token strings
ids = tokenizer.convert_tokens_to_ids(tokens)  # Tokens → integer IDs

print(f"Text:   '{text}'")
print(f"Tokens: {tokens}")
print(f"IDs:    {ids}")

# Decode back: IDs → text
decoded = tokenizer.decode(ids)
print(f"Decoded: '{decoded}'")

In [ ]:
# The __call__ method — the standard way to tokenize (adds special tokens)
# This is what you'll see in 95% of code

encoded = tokenizer("Hello, how are you?")
print("Encoded keys:", encoded.keys())
print("input_ids:     ", encoded['input_ids'])
print("attention_mask:", encoded['attention_mask'])
print("token_type_ids:", encoded['token_type_ids'])

# What each key means:
# input_ids       — the token IDs (what the model reads)
# attention_mask  — 1 for real tokens, 0 for padding
# token_type_ids  — which sentence (0 or 1) for BERT-style models

# Decode to see the special tokens
print("\nDecoded with special tokens:")
print(tokenizer.decode(encoded['input_ids']))
# [CLS] hello, how are you? [SEP]
# CLS = "start of sequence", SEP = "separator/end"

In [ ]:
# Batch tokenization with padding and truncation
# THE pattern you'll see in every training/inference script

texts = [
    "Short text.",
    "This is a much longer text that has more tokens in it.",
    "Medium length text here."
]

# Tokenize a batch — must pad to same length for batching
batch = tokenizer(
    texts,
    padding=True,          # Pad shorter sequences to match longest
    truncation=True,       # Truncate sequences longer than max_length
    max_length=20,         # Maximum sequence length
    return_tensors="pt"    # Return PyTorch tensors (use "tf" for TensorFlow, "np" for NumPy)
)

print("Batch shapes:")
print(f"  input_ids:      {batch['input_ids'].shape}")      # (3, 14) — 3 texts, padded
print(f"  attention_mask:  {batch['attention_mask'].shape}")

print("\nInput IDs (notice padding with 0s):")
print(batch['input_ids'])

print("\nAttention mask (0 = padding, 1 = real token):")
print(batch['attention_mask'])

In [ ]:
# Subword tokenization — why "playing" might become ["play", "##ing"]
# This is how transformers handle words they haven't seen before

words = ["playing", "unhappiness", "transformers", "supercalifragilistic"]
for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"  {word:25s} → {tokens}")

# Key facts about tokenizers:
# - Each model has its OWN tokenizer (BERT ≠ GPT ≠ T5)
# - ALWAYS use the tokenizer that matches the model
# - Subword tokenization means the vocab is ~30K-50K tokens
# - "##" prefix (BERT) = continuation of previous word
# - "Ġ" prefix (GPT) = starts with a space
# - Special tokens: [CLS], [SEP], [PAD], [UNK], <s>, </s>, <pad>

print(f"\nVocab size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print(f"Pad token ID:   {tokenizer.pad_token_id}")
print(f"CLS token ID:   {tokenizer.cls_token_id}")

## 4. Models — Loading and Using Pretrained Models

The `Auto*` classes automatically detect the right model architecture. Different tasks need different model heads.

In [ ]:
# AutoModel — loads the BASE model (no task-specific head)
# Returns raw hidden states — used for feature extraction / custom heads

model_name = "bert-base-uncased"
base_model = AutoModel.from_pretrained(model_name)

# Feed tokenized text through the model
inputs = tokenizer("The weather is beautiful today.", return_tensors="pt")

with torch.no_grad():
    outputs = base_model(**inputs)

# outputs contains the model's hidden states
print("Output keys:", outputs.keys())
print("last_hidden_state:", outputs.last_hidden_state.shape)
# (batch=1, seq_len=8, hidden_dim=768)
# Each token gets a 768-dimensional embedding

# The [CLS] token embedding is commonly used as a "sentence embedding"
cls_embedding = outputs.last_hidden_state[:, 0, :]  # First token = [CLS]
print("CLS embedding shape:", cls_embedding.shape)   # (1, 768)

In [ ]:
# Task-specific model classes — model + task head
# These are what you'll use for specific tasks

# Classification: AutoModelForSequenceClassification
# Adds a linear layer on top of [CLS] → num_labels outputs
cls_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3  # 3 classes
)

inputs = tokenizer("This is a test sentence.", return_tensors="pt")
with torch.no_grad():
    outputs = cls_model(**inputs)

print("Classification logits:", outputs.logits.shape)    # (1, 3)
print("Logits:", outputs.logits)

# Apply softmax to get probabilities
probs = torch.softmax(outputs.logits, dim=-1)
print("Probabilities:", probs)
print("Predicted class:", torch.argmax(probs, dim=-1).item())

# Common Auto*ForTask classes:
# AutoModelForSequenceClassification  — sentiment, topic classification
# AutoModelForTokenClassification     — NER, POS tagging
# AutoModelForQuestionAnswering       — extractive QA
# AutoModelForCausalLM                — text generation (GPT-style)
# AutoModelForSeq2SeqLM               — translation, summarization (T5, BART)
# AutoModelForMaskedLM                — fill-in-the-blank (BERT)

In [ ]:
# Model inspection — understanding what's inside

print(f"Model type: {type(cls_model).__name__}")
print(f"Config: {cls_model.config.model_type}")
print(f"Hidden size: {cls_model.config.hidden_size}")
print(f"Num layers: {cls_model.config.num_hidden_layers}")
print(f"Num attention heads: {cls_model.config.num_attention_heads}")
print(f"Vocab size: {cls_model.config.vocab_size}")

# Count parameters
total_params = sum(p.numel() for p in cls_model.parameters())
trainable = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

## 5. Model + Tokenizer Together — The Standard Pattern

This is the pattern you'll see in every real project: tokenize → model → post-process.

In [ ]:
# THE standard inference pattern — you'll see this everywhere

# Step 1: Load tokenizer + model (must match!)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Step 2: Tokenize
texts = ["I absolutely love this!", "This is terrible and I hate it."]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# Step 3: Forward pass (no grad for inference)
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Step 4: Post-process
probabilities = torch.softmax(outputs.logits, dim=-1)
predictions = torch.argmax(probabilities, dim=-1)

# Step 5: Map to labels
id2label = model.config.id2label  # {0: 'NEGATIVE', 1: 'POSITIVE'}
for text, pred, probs in zip(texts, predictions, probabilities):
    label = id2label[pred.item()]
    confidence = probs[pred].item()
    print(f"'{text}'")
    print(f"  → {label} ({confidence:.2%})\n")

In [ ]:
# Getting embeddings — used for semantic search, clustering, similarity

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

sentences = [
    "The cat sat on the mat.",
    "A kitten was sitting on a rug.",
    "The stock market crashed today."
]

# Tokenize
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

# Get embeddings
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Mean pooling — average all token embeddings (better than just CLS)
attention_mask = inputs['attention_mask'].unsqueeze(-1)  # (batch, seq, 1)
token_embeddings = outputs.last_hidden_state              # (batch, seq, 768)
sum_embeddings = (token_embeddings * attention_mask).sum(dim=1)
count = attention_mask.sum(dim=1)
mean_embeddings = sum_embeddings / count

print("Embedding shape:", mean_embeddings.shape)  # (3, 768)

# Cosine similarity — how similar are the sentences?
from torch.nn.functional import cosine_similarity
sim_01 = cosine_similarity(mean_embeddings[0:1], mean_embeddings[1:2]).item()
sim_02 = cosine_similarity(mean_embeddings[0:1], mean_embeddings[2:3]).item()
print(f"\n'cat on mat' vs 'kitten on rug':  {sim_01:.4f} (similar meaning)")
print(f"'cat on mat' vs 'stock market':   {sim_02:.4f} (different topic)")

## 6. Fine-Tuning with Hugging Face `Trainer`

Fine-tuning = taking a pretrained model and training it further on YOUR data. The `Trainer` class handles the training loop, evaluation, logging, and checkpointing.

In [ ]:
# Step 1: Prepare dataset — must be a HF Dataset or compatible format
from datasets import Dataset

# Simulate a sentiment dataset
train_texts = [
    "This product is amazing!", "Worst purchase ever.", "Pretty decent quality.",
    "I love it so much!", "Total waste of money.", "It works fine, nothing special.",
    "Absolutely fantastic!", "Horrible, do not buy.", "Good value for the price.",
    "Best thing I've bought!", "Broken on arrival.", "Meets expectations."
] * 5  # Repeat for more data

train_labels = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1] * 5

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
eval_dataset = Dataset.from_dict({
    "text": ["Great product!", "Terrible quality.", "It's okay I guess."],
    "label": [1, 0, 1]
})

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")
print(f"Sample:     {train_dataset[0]}")

In [ ]:
# Step 2: Tokenize the dataset
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# .map() applies the function to every example (like pandas .apply)
train_tokenized = train_dataset.map(tokenize_function, batched=True)
eval_tokenized = eval_dataset.map(tokenize_function, batched=True)

print("Tokenized columns:", train_tokenized.column_names)
print("Sample keys:", {k: type(v) for k, v in train_tokenized[0].items()})

In [ ]:
# Step 3: Load model + define training arguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

training_args = TrainingArguments(
    output_dir="./results",           # Where to save checkpoints
    num_train_epochs=2,               # Number of training epochs
    per_device_train_batch_size=8,    # Batch size per device
    per_device_eval_batch_size=8,
    eval_strategy="epoch",            # Evaluate at end of each epoch
    save_strategy="epoch",            # Save checkpoint each epoch
    logging_steps=10,                 # Log every N steps
    learning_rate=2e-5,               # Fine-tuning LR (much smaller than training from scratch)
    weight_decay=0.01,                # L2 regularization
    load_best_model_at_end=True,      # Load the best checkpoint when done
    report_to="none",                 # Disable wandb/tensorboard for this demo
)

print("Training arguments configured")

In [ ]:
# Step 4: Define metrics and create Trainer
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    compute_metrics=compute_metrics,
)

# Step 5: Train!
trainer.train()

# Step 6: Evaluate
results = trainer.evaluate()
print(f"\nEval results: {results}")

## 7. Fine-Tuning with Native PyTorch

When `Trainer` isn't flexible enough, you write the training loop yourself. This is the same PyTorch pattern from the PyTorch notebook, adapted for transformers.

In [ ]:
# Native PyTorch fine-tuning — for when you need full control
from torch.utils.data import DataLoader

# Prepare data
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Create a simple PyTorch dataset from our tokenized HF dataset
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
train_loader = DataLoader(train_tokenized, batch_size=8, shuffle=True)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Training loop — same 5 steps as vanilla PyTorch
model.train()
for epoch in range(2):
    total_loss = 0
    for batch in train_loader:
        # HF models compute loss internally when labels are provided!
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["label"]          # Pass labels → model returns loss
        )
        loss = outputs.loss                 # Loss is computed inside the model
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: avg loss = {avg_loss:.4f}")

# KEY INSIGHT: HF models compute the loss FOR YOU when you pass labels=
# This is different from vanilla PyTorch where you compute loss separately

## 8. Text Generation with LLMs

How GPT-style models generate text. You'll see these patterns in chatbots, code assistants, and content generators.

In [ ]:
# Text generation with a small GPT-2 model
generator = pipeline("text-generation", model="distilgpt2")

result = generator(
    "The future of artificial intelligence is",
    max_new_tokens=50,       # How many tokens to generate
    num_return_sequences=2,  # Generate multiple completions
    temperature=0.7,         # Randomness (0=deterministic, 1=creative)
    do_sample=True,          # Enable sampling (vs greedy)
)

print("Text generation results:")
for i, r in enumerate(result):
    print(f"\n--- Completion {i+1} ---")
    print(r['generated_text'])

In [ ]:
# Lower-level generation with model.generate() — more control
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Must set pad token for batched generation
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

prompt = "Machine learning is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generation strategies you'll see in AI code:
# 1. Greedy — always pick the most likely next token
greedy = model.generate(**inputs, max_new_tokens=30, do_sample=False)
print("Greedy:", tokenizer.decode(greedy[0], skip_special_tokens=True))

# 2. Sampling with temperature — random but controlled
sampled = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=0.8)
print("\nSampled:", tokenizer.decode(sampled[0], skip_special_tokens=True))

# 3. Top-k sampling — only sample from top k tokens
topk = model.generate(**inputs, max_new_tokens=30, do_sample=True, top_k=50)
print("\nTop-k:", tokenizer.decode(topk[0], skip_special_tokens=True))

# 4. Top-p (nucleus) sampling — sample from smallest set with cumulative prob > p
topp = model.generate(**inputs, max_new_tokens=30, do_sample=True, top_p=0.9)
print("\nTop-p:", tokenizer.decode(topp[0], skip_special_tokens=True))

## 9. Saving, Loading, and the Model Hub

In [ ]:
# Save model and tokenizer locally
# model.save_pretrained("./my-model")
# tokenizer.save_pretrained("./my-model")

# Load back
# model = AutoModelForSequenceClassification.from_pretrained("./my-model")
# tokenizer = AutoTokenizer.from_pretrained("./my-model")

# Push to Hugging Face Hub (share with the world)
# model.push_to_hub("my-username/my-model-name")
# tokenizer.push_to_hub("my-username/my-model-name")

# Load from the Hub (anyone can use your model now)
# model = AutoModelForSequenceClassification.from_pretrained("my-username/my-model-name")

# Popular models on the Hub:
# "bert-base-uncased"                              — BERT (NLU tasks)
# "distilbert-base-uncased"                        — Smaller, faster BERT
# "roberta-base"                                   — Improved BERT
# "gpt2" / "distilgpt2"                            — GPT-2 (text generation)
# "meta-llama/Llama-2-7b-hf"                       — LLaMA 2 (needs access)
# "google/flan-t5-base"                            — T5 (text-to-text)
# "sentence-transformers/all-MiniLM-L6-v2"         — Sentence embeddings
# "facebook/bart-large-mnli"                       — Zero-shot classification

print("Model Hub: https://huggingface.co/models")
print("(Save/push commands commented out to avoid file creation)")

## 10. Advanced Patterns — LoRA, Quantization, and Custom Heads

Techniques you'll see in cutting-edge AI code for efficient fine-tuning and deployment.

In [ ]:
# Pattern 1: Custom classification head on a pretrained backbone
# When the built-in AutoModelForSequenceClassification isn't enough

import torch.nn as nn

class CustomClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # Custom head — more complex than the default single linear layer
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        logits = self.classifier(cls_embedding)
        
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        
        return {"loss": loss, "logits": logits}

custom_model = CustomClassifier("distilbert-base-uncased", num_labels=5)
print(f"Custom model params: {sum(p.numel() for p in custom_model.parameters()):,}")

In [ ]:
# Pattern 2: Freeze backbone, train only the head
# The most common transfer learning approach

# Freeze the pretrained backbone
for param in custom_model.backbone.parameters():
    param.requires_grad = False

# Only the classifier head will be trained
trainable = sum(p.numel() for p in custom_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in custom_model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")
print("Only the custom head is trainable — backbone is frozen")

In [ ]:
# Pattern 3: LoRA (Low-Rank Adaptation) — efficient fine-tuning
# Trains only small adapter matrices instead of full model weights
# Requires: pip install peft

# from peft import LoraConfig, get_peft_model, TaskType
#
# model = AutoModelForSequenceClassification.from_pretrained(
#     "bert-base-uncased", num_labels=2
# )
#
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,
#     r=8,                    # Rank of adaptation matrices (lower = fewer params)
#     lora_alpha=32,          # Scaling factor
#     lora_dropout=0.1,
#     target_modules=["query", "value"],  # Which attention matrices to adapt
# )
#
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()
# # trainable params: ~300K vs ~110M total (0.27%)

print("LoRA pattern (requires 'peft' library):")
print("  - Trains ~0.1-1% of parameters")
print("  - Same quality as full fine-tuning for many tasks")
print("  - Can be merged back into base model for deployment")

In [ ]:
# Pattern 4: Quantization — shrink model size for deployment
# Load model in 8-bit or 4-bit precision (requires: pip install bitsandbytes)

# 8-bit quantization (reduces memory ~2x)
# model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-2-7b-hf",
#     load_in_8bit=True,
#     device_map="auto"
# )

# 4-bit quantization (reduces memory ~4x) — used for running 7B+ models on consumer GPUs
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_quant_type="nf4",
# )
# model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-2-7b-hf",
#     quantization_config=quantization_config,
#     device_map="auto"
# )

print("Quantization patterns (requires GPU + bitsandbytes):")
print("  8-bit: load_in_8bit=True         → ~2x memory reduction")
print("  4-bit: BitsAndBytesConfig(4bit)   → ~4x memory reduction")
print("  Often combined with LoRA: quantized base + LoRA adapters (QLoRA)")

## 11. The `datasets` Library — Loading and Processing Data

The companion `datasets` library is tightly integrated with `transformers`. You'll see it in every fine-tuning project.

In [ ]:
from datasets import load_dataset, Dataset

# Load a dataset from the Hugging Face Hub
# dataset = load_dataset("imdb")                    # Full IMDB reviews
# dataset = load_dataset("squad")                   # SQuAD QA
# dataset = load_dataset("glue", "mrpc")            # GLUE benchmark
# dataset = load_dataset("csv", data_files="data.csv")  # From local CSV

# Create from dict (what we did earlier)
ds = Dataset.from_dict({
    "text": ["Hello world", "How are you", "Fine thanks"],
    "label": [0, 1, 1]
})

print("Dataset:", ds)
print("Features:", ds.features)
print("First row:", ds[0])

# Common operations:
print("\n.select([0,2]):", ds.select([0, 2]))         # Select specific indices
print(".filter:", ds.filter(lambda x: x['label'] == 1))  # Filter rows
print(".shuffle:", ds.shuffle(seed=42))

# .map() — transform every example (for tokenization)
def uppercase(example):
    example['text'] = example['text'].upper()
    return example

ds_upper = ds.map(uppercase)
print("\n.map (uppercase):", ds_upper['text'])

# Train/test split
split = ds.train_test_split(test_size=0.33, seed=42)
print(f"\nSplit: train={len(split['train'])}, test={len(split['test'])}")

## Quick Reference

### Core Pattern — What You'll See in Every Project
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("model-name")
model = AutoModelForSequenceClassification.from_pretrained("model-name")

inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
outputs = model(**inputs)
```

### Auto Classes — Which One to Use

| Task | Auto Class | Output |
|---|---|---|
| Feature extraction / embeddings | `AutoModel` | Hidden states |
| Text classification (sentiment, topic) | `AutoModelForSequenceClassification` | Logits per class |
| Token classification (NER, POS) | `AutoModelForTokenClassification` | Logits per token |
| Question answering | `AutoModelForQuestionAnswering` | Start/end logits |
| Text generation (GPT) | `AutoModelForCausalLM` | Next token logits |
| Seq2Seq (T5, BART) | `AutoModelForSeq2SeqLM` | Generated token IDs |
| Fill mask (BERT) | `AutoModelForMaskedLM` | Logits per position |

### Tokenizer Cheat Sheet

| What | Code | Notes |
|---|---|---|
| Load | `AutoTokenizer.from_pretrained("model")` | Must match the model |
| Tokenize | `tokenizer(text, return_tensors="pt")` | Returns dict of tensors |
| Batch + pad | `tokenizer(texts, padding=True, truncation=True)` | For batches |
| Decode | `tokenizer.decode(ids, skip_special_tokens=True)` | IDs → text |
| Vocab size | `tokenizer.vocab_size` | |
| Special tokens | `tokenizer.all_special_tokens` | `[CLS]`, `[SEP]`, etc. |

### Generation Parameters

| Parameter | What it does | Typical values |
|---|---|---|
| `max_new_tokens` | Max tokens to generate | 50-500 |
| `temperature` | Randomness (lower=focused) | 0.1-1.0 |
| `top_k` | Sample from top k tokens | 50 |
| `top_p` | Nucleus sampling threshold | 0.9-0.95 |
| `do_sample` | Enable sampling (vs greedy) | True/False |
| `num_beams` | Beam search width | 4-5 |
| `repetition_penalty` | Penalize repeated tokens | 1.1-1.5 |

### Essential Vocabulary

| Term | Meaning |
|---|---|
| **Pretrained model** | Model trained on massive data (Wikipedia, books, web) |
| **Fine-tuning** | Training a pretrained model on your specific task/data |
| **Tokenizer** | Converts text ↔ token IDs |
| **Pipeline** | High-level API: tokenizer + model + post-processing |
| **[CLS] token** | Special "classification" token (BERT-style, first position) |
| **Attention mask** | 1 = real token, 0 = padding (tells model what to attend to) |
| **Logits** | Raw model output (before softmax) |
| **Backbone** | The pretrained body of the model (without task head) |
| **Head** | Task-specific layer(s) on top of the backbone |
| **LoRA** | Efficient fine-tuning — trains tiny adapter matrices |
| **Quantization** | Reduce precision (float32→int8/int4) for deployment |
| **Hub** | huggingface.co — hosting for models, datasets, and spaces |